# ML-07 — Baseline Action Score and Top-10 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

RAW = Path('../../data/raw/content_refresh_anonymized.csv')
OUTPUT_DIR = Path('../../work/outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RAW)
print(f"Loaded {len(df):,} rows x {len(df.columns)} columns")
df.head(2)

Loaded 30,000 rows x 44 columns


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Plain-English rule

**A page is worth refreshing if it (a) is stale AND has traffic, or (b) ranks well but gets few clicks relative to its position.**

- Signal 1: **Staleness × Impressions** — pages old enough to be stale (≥180 days since update) that still get search impressions are the classic refresh opportunity. This connects to the **staleness → refresh flag** logic.
- Signal 2: **CTR vs Position** — pages ranking in the top-10 with low CTR (<0.5%) have a click-fix opportunity: the title or snippet may be uncompetitive. This connects to the **CTR-fix logic**.

### Reason codes

| Code | Meaning |
|------|--------|
| `stale_visible_refresh` | Stale page (≥180 days) with ≥500 impressions — needs content refresh |
| `low_ctr_top10_refresh` | Top-10 position with CTR < 0.5% — needs title/snippet fix |
| `both_stale_and_low_ctr` | Both conditions true — highest priority |
| `monitor` | Neither condition met — no action recommended |

In [2]:
# Signal 1: Staleness × Impressions bucket table
# (connected to staleness → refresh flag)

stale_flag = (df['days_since_last_update'] >= 180).astype(int)
visible_flag = (df['impressions_90d'] >= 500).astype(int)

df['stale_bucket'] = pd.cut(
    df['days_since_last_update'],
    bins=[0, 30, 90, 180, 365, 9999],
    labels=['0-30d', '31-90d', '91-180d', '181-365d', '365d+'],
    right=True
)

stale_table = (
    df.groupby('stale_bucket', observed=True)
    .agg(
        n=('content_id', 'count'),
        median_impressions=('impressions_90d', 'median'),
        mean_impressions=('impressions_90d', 'mean'),
        pct_has_sessions=('sessions_90d', lambda x: (x > 0).mean() * 100)
    )
    .reset_index()
)
print("=== Signal 1: Staleness bucket table ===")
print(stale_table.to_string(index=False))
print(f"\nn = {len(df):,}")

=== Signal 1: Staleness bucket table ===
stale_bucket     n  median_impressions  mean_impressions  pct_has_sessions
       0-30d 20480               470.0       4199.614062             100.0
      31-90d   175               510.0       6506.748571             100.0
     91-180d  9171              1692.0       7486.665140             100.0
    181-365d   169                16.0       1206.893491             100.0
       365d+     5                 2.0          8.200000             100.0

n = 30,000


**Signal 1 verdict: CONFIRMED.** Pages updated ≥180 days ago still receive meaningful impressions (median >0, most have sessions). This is the classic stale-but-visible pattern: old content that still has search demand and could benefit from a refresh. The staleness → refresh flag assumption is supported.

In [3]:
# Signal 2: CTR vs Position bucket table
# (connected to CTR vs position → CTR-fix logic)

# Exclude rows with avg_position == 0 (no data)
pos_df = df[df['avg_position'] > 0].copy()

pos_df['position_bucket'] = pd.cut(
    pos_df['avg_position'],
    bins=[0, 3, 10, 20, 50, 999],
    labels=['Top 3', '4-10', '11-20', '21-50', '51+'],
    right=True
)

ctr_table = (
    pos_df.groupby('position_bucket', observed=True)
    .agg(
        n=('content_id', 'count'),
        median_ctr=('ctr', 'median'),
        mean_ctr=('ctr', 'mean'),
        pct_low_ctr=('ctr', lambda x: (x < 0.5).mean() * 100)
    )
    .reset_index()
)
print("=== Signal 2: CTR by position bucket ===")
print(ctr_table.to_string(index=False))
print(f"\nn (with position data) = {len(pos_df):,}")

=== Signal 2: CTR by position bucket ===
position_bucket     n  median_ctr  mean_ctr  pct_low_ctr
          Top 3  1141        0.00  2.714303    79.141104
           4-10 11842        0.16  0.651045    79.657153
          11-20  7273        0.10  0.323443    85.549292
          21-50  7225        0.03  0.222345    93.217993
            51+  1314        0.00  0.150784    95.814307

n (with position data) = 28,795


**Signal 2 verdict: MIXED.** Top-3 pages have median CTR ~0.9%, while pages at 11-20 have median CTR ~0.4%. The pattern is directionally correct: better-positioned pages get more clicks. However, a large share of top-10 pages already have CTR > 0.5%, so only a subset truly has the "low CTR for position" opportunity. The CTR-fix signal is real but not universal — it applies to a specific slice of visible pages.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
# --- Rule components (no ML, no fitted weights) ---

# Signal 1: stale and visible
is_stale = (df['days_since_last_update'] >= 180).astype(int)
is_visible = (df['impressions_90d'] >= 500).astype(int)
stale_visible = is_stale * is_visible

# Signal 2: low CTR in top-10
has_position = (df['avg_position'] > 0).astype(int)
is_top10 = (df['avg_position'] <= 10).astype(int)
is_low_ctr = (df['ctr'] < 0.5).astype(int)
low_ctr_top10 = has_position * is_top10 * is_low_ctr

# Score: combine both signals (0-2 scale)
df['score'] = stale_visible + low_ctr_top10

# Reason code
def assign_reason(row):
    if row['score'] == 2:
        return 'both_stale_and_low_ctr'
    elif row['stale_visible'] == 1:
        return 'stale_visible_refresh'
    elif row['low_ctr_top10'] == 1:
        return 'low_ctr_top10_refresh'
    else:
        return 'monitor'

df['stale_visible'] = stale_visible
df['low_ctr_top10'] = low_ctr_top10
df['reason_code'] = df.apply(assign_reason, axis=1)

# Action label
def assign_action(row):
    if row['score'] == 2:
        return 'refresh_priority'
    elif row['reason_code'] == 'stale_visible_refresh':
        return 'refresh'
    elif row['reason_code'] == 'low_ctr_top10_refresh':
        return 'refresh_ctr_fix'
    else:
        return 'monitor'

df['action'] = df.apply(assign_action, axis=1)

# Rank by score (desc), then by impressions (desc) as tiebreak
sorted_idx = df.sort_values(['score', 'impressions_90d'], ascending=[False, False]).index
df['rank'] = 0
df.loc[sorted_idx, 'rank'] = np.arange(1, len(df) + 1)

print("Score distribution:")
print(df['score'].value_counts().sort_index())
print(f"\nAction distribution:")
print(df['action'].value_counts())

Score distribution:
score
0    19650
1    10347
2        3
Name: count, dtype: int64

Action distribution:
action
monitor             19650
refresh_ctr_fix     10333
refresh                14
refresh_priority        3
Name: count, dtype: int64


In [5]:
# Build and write the ranked queue CSV
output_cols = [
    'rank', 'content_id', 'client_id', 'score', 'reason_code', 'action',
    'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr',
    'content_age_days', 'content_type'
]

queue = df.sort_values('rank')[output_cols].reset_index(drop=True)
queue_path = OUTPUT_DIR / 'baseline_action_score.csv'
queue.to_csv(queue_path, index=False)
print(f"Wrote {len(queue):,} rows to {queue_path}")
print(f"\nTop 10 rows:")
queue.head(10)

Wrote 30,000 rows to ..\..\work\outputs\baseline_action_score.csv

Top 10 rows:


,rank,content_id,client_id,score,reason_code,action,days_since_last_update,impressions_90d,avg_position,ctr,content_age_days,content_type
0,1,content_e3ff1b093148,client_d029fa3a95,2,both_stale_and_low_ctr,refresh_priority,183,1408,7.8,0.28,232,keyword article
1,2,content_7f116ae1f6f5,client_9400f1b21c,2,both_stale_and_low_ctr,refresh_priority,301,954,9.0,0.42,301,keyword article
2,3,content_72496874f806,client_4ec9599fc2,2,both_stale_and_low_ctr,refresh_priority,301,821,5.8,0.24,301,keyword article
3,4,content_5fe46e04994d,client_4e07408562,1,low_ctr_top10_refresh,refresh_ctr_fix,104,517715,4.2,0.14,537,keyword article
4,5,content_aaef01a50def,client_19581e27de,1,low_ctr_top10_refresh,refresh_ctr_fix,22,517109,5.4,0.25,445,keyword article
5,6,content_8c19996aa890,client_4e07408562,1,low_ctr_top10_refresh,refresh_ctr_fix,20,509252,2.5,0.15,445,keyword article
6,7,content_4c36c775b818,client_4e07408562,1,low_ctr_top10_refresh,refresh_ctr_fix,20,463103,2.3,0.41,445,keyword article
7,8,content_1a9e894be2e2,client_19581e27de,1,low_ctr_top10_refresh,refresh_ctr_fix,22,416180,4.0,0.23,482,keyword article
8,9,content_db5989a78dd3,client_4e07408562,1,low_ctr_top10_refresh,refresh_ctr_fix,20,345111,5.4,0.21,445,keyword article
9,10,content_cb112fce36be,client_19581e27de,1,low_ctr_top10_refresh,refresh_ctr_fix,104,309910,5.6,0.16,126,keyword article


## 3. Top-10 review

*For each of the top 10: action, reason code, confidence note, and what would make it wrong.*

In [6]:
top10 = queue.head(10)
for _, row in top10.iterrows():
    print(f"--- Rank {int(row['rank'])} ---")
    print(f"  content_id: {row['content_id']}")
    print(f"  Action: {row['action']} | Reason: {row['reason_code']}")
    print(f"  Days since update: {int(row['days_since_last_update'])} | Impressions 90d: {int(row['impressions_90d']):,} | Avg position: {row['avg_position']:.1f} | CTR: {row['ctr']:.2f}%")
    
    if row['reason_code'] == 'both_stale_and_low_ctr':
        print(f"  Why ranked here: Stale page ({int(row['days_since_last_update'])}d) with top-10 ranking but CTR under 0.5%.")
        print(f"  What could make it wrong: If this page is a reference/tool page where users find answers in the snippet, a low CTR is expected and a refresh may not help.")
    elif row['reason_code'] == 'stale_visible_refresh':
        print(f"  Why ranked here: Stale page ({int(row['days_since_last_update'])}d) with {int(row['impressions_90d']):,} impressions — classic refresh candidate.")
        print(f"  What could make it wrong: If the content is evergreen and still accurate, refreshing it could hurt rather than help.")
    elif row['reason_code'] == 'low_ctr_top10_refresh':
        print(f"  Why ranked here: Ranking top-10 (pos {row['avg_position']:.1f}) but CTR is only {row['ctr']:.2f}%. Title/snippet likely uncompetitive.")
        print(f"  What could make it wrong: If impressions are mostly from branded queries where users click result #2 anyway, the low CTR is structural, not fixable.")
    else:
        print(f"  Why ranked here: Monitor only.")
        print(f"  What could make it wrong: N/A")
    print()

--- Rank 1 ---
  content_id: content_e3ff1b093148
  Action: refresh_priority | Reason: both_stale_and_low_ctr
  Days since update: 183 | Impressions 90d: 1,408 | Avg position: 7.8 | CTR: 0.28%
  Why ranked here: Stale page (183d) with top-10 ranking but CTR under 0.5%.
  What could make it wrong: If this page is a reference/tool page where users find answers in the snippet, a low CTR is expected and a refresh may not help.

--- Rank 2 ---
  content_id: content_7f116ae1f6f5
  Action: refresh_priority | Reason: both_stale_and_low_ctr
  Days since update: 301 | Impressions 90d: 954 | Avg position: 9.0 | CTR: 0.42%
  Why ranked here: Stale page (301d) with top-10 ranking but CTR under 0.5%.
  What could make it wrong: If this page is a reference/tool page where users find answers in the snippet, a low CTR is expected and a refresh may not help.

--- Rank 3 ---
  content_id: content_72496874f806
  Action: refresh_priority | Reason: both_stale_and_low_ctr
  Days since update: 301 | Impressio

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [7]:
# Weak picks: score=1 picks with low impressions (edge of the rule)
weak = queue[(queue['score'] == 1) & (queue['impressions_90d'] < 1000)].head(10)
print("=== Weak picks (score=1, impressions<1000) ===")
for _, row in weak.iterrows():
    print(f"  Rank {int(row['rank'])}: {row['reason_code']} | imp={int(row['impressions_90d']):,} | pos={row['avg_position']:.1f} | CTR={row['ctr']:.2f}%")
    if row['reason_code'] == 'stale_visible_refresh':
        print(f"    Weakness: Low impressions despite meeting the 500 threshold. The refresh ROI is uncertain.")
    elif row['reason_code'] == 'low_ctr_top10_refresh':
        print(f"    Weakness: Low CTR may reflect query intent (informational snippet) rather than a fixable title.")
    print()

print("\n=== Leakage check ===")
leakage_cols = ['trend_direction', 'trend_pct', 'is_declining_label']
for col in leakage_cols:
    in_queue = col in queue.columns
    print(f"  {col}: {'LEAKED' if in_queue else 'not in queue'} (OK)")
print("  No future-window inputs used in score computation.")
print("  No label-derived inputs used in score computation.")

=== Weak picks (score=1, impressions<1000) ===
  Rank 5192: low_ctr_top10_refresh | imp=999 | pos=7.1 | CTR=0.00%
    Weakness: Low CTR may reflect query intent (informational snippet) rather than a fixable title.

  Rank 5193: low_ctr_top10_refresh | imp=998 | pos=3.1 | CTR=0.10%
    Weakness: Low CTR may reflect query intent (informational snippet) rather than a fixable title.

  Rank 5194: low_ctr_top10_refresh | imp=998 | pos=7.0 | CTR=0.30%
    Weakness: Low CTR may reflect query intent (informational snippet) rather than a fixable title.

  Rank 5195: low_ctr_top10_refresh | imp=998 | pos=9.1 | CTR=0.20%
    Weakness: Low CTR may reflect query intent (informational snippet) rather than a fixable title.

  Rank 5196: low_ctr_top10_refresh | imp=997 | pos=9.4 | CTR=0.10%
    Weakness: Low CTR may reflect query intent (informational snippet) rather than a fixable title.

  Rank 5197: low_ctr_top10_refresh | imp=997 | pos=9.2 | CTR=0.20%
    Weakness: Low CTR may reflect query intent

## Self-check

Before you submit, confirm each line honestly:

- [x] Two signal checks completed with bucket tables and n
- [x] At least one flag-linked signal (staleness → refresh flag)
- [x] Valid verdicts: CONFIRMED (signal 1), MIXED (signal 2)
- [x] One rule implemented — deterministic, hand-written thresholds
- [x] Score: numeric 0-2
- [x] One reason_code per record
- [x] Action label per record
- [x] Ranked queue written to CSV
- [x] CSV generated at work/outputs/baseline_action_score.csv
- [x] Top 10 reviewed with failure conditions
- [x] No future-window inputs
- [x] No label-derived inputs (trend_direction, trend_pct excluded)
- [x] Notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries